<a href="https://colab.research.google.com/github/Maziger/Laksegate-master-thesis/blob/main/POC/FINSPID_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FINSPID (Financial News and Stock Price Integration Dataset)

github: https://github.com/Zdong104/FNSPID_Financial_News_Dataset

Hugging face: https://huggingface.co/papers/2402.06698

Paper: https://arxiv.org/pdf/2402.06698

In [ ]:
!git clone https://github.com/Zdong104/FNSPID_Financial_News_Dataset.git

Cloning into 'FNSPID_Financial_News_Dataset'...
remote: Enumerating objects: 1511, done.
remote: Counting objects: 100% (197/197), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 1511 (delta 162), reused 120 (delta 120), pack-reused 1314 (from 1)
Receiving objects: 100% (1511/1511), 105.90 MiB | 20.74 MiB/s, done.
Resolving deltas: 100% (465/465), done.
Updating files: 100% (1083/1083), done.


In [ ]:
%cd FNSPID_Financial_News_Dataset
!pip install -r requirements.txt

/content/FNSPID_Financial_News_Dataset
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 69.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [ ]:
# Download takes ~5 minutes, so maybe get a cup of coffee or do some push-ups
from datasets import load_dataset

dataset = load_dataset('Zihan1004/FNSPID', data_files='Stock_price/full_history.zip')

Repo card metadata block was not found. Setting CardData to empty.


Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
import pandas as pd

okdokey = dataset['train'][0:1000]
okdokey = pd.DataFrame(dataset['train'])
okdokey


In [ ]:
okdokey

NameError: name 'okdokey' is not defined

In [ ]:
!unzip Stock_price/full_history.zip -d full_history

In [ ]:
import pandas as pd
news_data = pd.read_csv('All_external.csv')
market_data = pd.read_csv('All_market.csv')

In [ ]:
dataset

NameError: name 'dataset' is not defined

# Classifying news using GLiClass Instruct

Hugging Face: https://huggingface.co/knowledgator/gliclass-instruct-large-v1.0

In [1]:
!pip install -q gliclass

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.8 MB/s eta 0:00:00


In [16]:
pipeline(['what is love?', 'big tech is so good!'], labels=['energy', 'tech'], threshold=0.02)

100%|██████████| 1/1 [00:00<00:00,  9.62it/s]


[[{'label': 'energy', 'score': 0.805618941783905},
  {'label': 'tech', 'score': 0.21098583936691284}],
 [{'label': 'energy', 'score': 0.09507032483816147}]]

In [2]:
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from transformers import AutoTokenizer

model = GLiClassModel.from_pretrained("knowledgator/gliclass-instruct-large-v1.0")
tokenizer = AutoTokenizer.from_pretrained("knowledgator/gliclass-instruct-large-v1.0")
pipeline = ZeroShotClassificationPipeline(model, tokenizer, classification_type='multi-label', device='cuda:0')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.75G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/92.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

In [6]:
text = "TTF prices up 2% this morning because of oil"
labels = ["energy", "finance", "tech", "health"]

# examples = [
#     {"text": "TTF prices are up by 5%.", "labels": ["energy"]},
#     {"text": "Boycots will affect the oil prices.", "labels": ["energy"]},
#     {"text": "CEO of Starbucks is retiering.", "labels": ["not energy"]},
# ]

results = pipeline(text, labels, threshold=0.02)[0]
print()
for r in results:
    print(r["label"], "=>", r["score"])

100%|██████████| 1/1 [00:00<00:00,  1.88it/s]


energy => 0.7042527794837952
finance => 0.7362023591995239
tech => 0.11519643664360046


In [7]:
import pandas as pd
data = pd.read_csv('nasdaq_exteral_data_Y.csv')
data = data.set_index('Unnamed: 0')
data.index.name = None
data = data[data.index.notna()]
data.head()

,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
2463320.0,2023-12-15 00:00:00 UTC,Worsening Deflation Pushes Inverse China ETF H...,YANG,https://www.nasdaq.com/articles/worsening-defl...,NaN,NaN,Worsening deflation is putting the Direxion Da...,"With YINN down 25%, traders can take the other...","With YINN down 25%, traders can take the other...","With YINN down 25%, traders can take the other...","With YINN down 25%, traders can take the other..."
2463321.0,2023-12-14 00:00:00 UTC,Top Performing Leveraged/Inverse ETFs: 12/10/2023,YANG,https://www.nasdaq.com/articles/top-performing...,NaN,NaN,Top Performing Levered/Inverse ETFs Last Week\...,YANG – Direxion Daily FTSE China Bear 3X Share...,(GDXD ) MicroSectors Gold Miners -3X Inverse L...,(GDXD ) MicroSectors Gold Miners -3X Inverse L...,(GDXD ) MicroSectors Gold Miners -3X Inverse L...
2463322.0,2023-12-13 00:00:00 UTC,Best Inverse/Leveraged ETFs of Last Week,YANG,https://www.nasdaq.com/articles/best-inverse-l...,NaN,NaN,The S&P 500 hit a new high for the year last w...,Direxion Daily FTSE China Bear 3X Shares YANG ...,Click to get this free report United States Oi...,Click to get this free report United States Oi...,Click to get this free report United States Oi...
2463323.0,2023-12-05 00:00:00 UTC,Top Performing Leveraged/Inverse ETFs: 12/03/2023,YANG,https://www.nasdaq.com/articles/top-performing...,NaN,NaN,These were last week's top performing leverage...,(GDXU B-) MicroSectors Gold Miners 3X Leverage...,(GDXU B-) MicroSectors Gold Miners 3X Leverage...,(GDXU B-) MicroSectors Gold Miners 3X Leverage...,(GDXU B-) MicroSectors Gold Miners 3X Leverage...
2463324.0,2023-11-01 00:00:00 UTC,Get in Early on China’s Economic Recovery With...,YANG,https://www.nasdaq.com/articles/get-in-early-o...,NaN,NaN,The Chinese government is injecting a dose of ...,"Additionally, if investor sentiment remains be...","Additionally, if investor sentiment remains be...","Additionally, if investor sentiment remains be...","Additionally, if investor sentiment remains be..."


In [8]:
data_small = data.iloc[:1000]
data_small = data_small['Lsa_summary']
# data_small

In [23]:
data_small_list = data_small.to_list()

In [26]:
result_test = pipeline(data_small_list, labels, threshold=0.5)


100%|██████████| 125/125 [01:12<00:00,  1.73it/s]


In [9]:
classified_labels = []
classified_scores = []

for text_to_classify in data_small:
    results = pipeline(text_to_classify, labels, threshold=0.5)[0]
    # Extract labels and scores
    flag = any(result['label'] == 'energy' for result in results)
    if flag:
        classified_labels.append('energy')
    else:
        classified_labels.append('not energy')

# Add the classified data back to the data_small DataFrame
res = pd.DataFrame(data_small) # Ensure data_small is a DataFrame to add columns
res['classified_labels'] = classified_labels

res.head()

100%|██████████| 1/1 [00:00<00:00, 17.19it/s]


,Lsa_summary,classified_labels
2463320.0,"With YINN down 25%, traders can take the other...",not energy
2463321.0,YANG – Direxion Daily FTSE China Bear 3X Share...,not energy
2463322.0,Direxion Daily FTSE China Bear 3X Shares YANG ...,energy
2463323.0,(GDXU B-) MicroSectors Gold Miners 3X Leverage...,not energy
2463324.0,"Additionally, if investor sentiment remains be...",not energy


In [11]:
len(res[res['classified_labels'] == 'energy'])

34

In [125]:
res[res['classified_labels'] == 'energy'].index

Index([2466322.0, 2466331.0, 2466332.0, 2466338.0, 2466346.0, 2466356.0,
       2466357.0, 2466358.0, 2466364.0, 2466368.0, 2466369.0, 2466370.0,
       2466371.0, 2466372.0, 2466373.0, 2466378.0, 2466379.0, 2466387.0,
       2466403.0, 2466407.0, 2466414.0, 2466417.0],
      dtype='float64')

In [123]:
for article in res[res['classified_labels'] == 'energy']['Lsa_summary']:
    print(article)
    print('\n')

By Stephanie Kelly, Jarrett Renshaw and Leah Douglas NEW YORK, Dec 14 (Reuters) - The Biden administration is expected this week to recognize a soon-to-be-updated methodology favored by the ethanol industry in guidance to companies looking to claim tax credits for sustainable aviation fuel (SAF), three people familiar with the matter told Reuters. As it stands, that model would enable ethanol-based SAF to qualify for tax credits under the Inflation Reduction Act, President Joe Biden's signature climate law. One of environmental groups' central concerns about the use of the GREET model - that it would underestimate the emissions generated by tilling land for crops - appears to be unresolved, said Nikita Pavlenko, fuels team lead at the International Council on Clean Transportation.


Front-month gas futures NGc1 for January delivery on the New York Mercantile Exchange were up 1.2 cents, or 0.5%, to $2.35 per million British thermal units (mmBtu) at 12:45 p.m. EST (1745 GMT). "We will re

In [ ]:
any(result == 'energy' for result in results)